# Experiment 10 — Multi-seed MLP Ensemble in 3-Way Blend

**Builds on:** Exp09 (3-way blend cmAP 0.9144; best: 10% MLP + 60% tuned_LR + 30% raw Perch + 5% prior).

**Hypothesis:** The MLP component has high fold-level AUC variance (0.77–0.92 in exp09). Averaging 3 independent MLP seeds reduces this variance and may allow a larger MLP weight in the blend, improving cmAP further.

**Design:**
- Train 3 MLPs per fold (seeds 42, 123, 456), average their OOF predictions
- Same LR probes as exp09 (C=0.05, balanced, PCA64)
- Finer 3-way blend grid concentrated around exp09 best (a_mlp~0.10-0.30, a_lr~0.50-0.70)
- Same site/hour priors on top

In [1]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.special import expit as sigmoid
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
DEVICE = 'cpu'

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
CACHE_DIR = DATA_DIR / 'cache'

EMB_CACHE = CACHE_DIR / 'exp02_embeddings.npy'
LOG_CACHE = CACHE_DIR / 'exp02_logits.npy'
RESULTS_CSV = CACHE_DIR / 'exp10_mlp_ensemble_blend_results.csv'
RESULTS_JSON = CACHE_DIR / 'exp10_mlp_ensemble_blend_best.json'

N_SPLITS = 5
PCA_DIM = 64
MIN_POS = 3
LR_C = 0.05
PROBE_BLEND = 0.25
PRIOR_BLEND_GRID = [0.00, 0.025, 0.05, 0.075, 0.10]

MLP_SEEDS = [42, 123, 456]
HIDDEN = [512, 256]
DROPOUT = 0.4
MLP_LR = 3e-4
WEIGHT_DECAY = 1e-3
MAX_EPOCHS = 300
PATIENCE = 25
BATCH_SIZE = 64
POS_WEIGHT_CAP = 30.0

# Finer grid concentrated around exp09 best (a_mlp~0.1, a_lr~0.6, a_perch~0.3)
A_MLP_GRID = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
A_LR_GRID  = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

print('Project root:', PROJECT_ROOT)

Project root: /Users/jroessler/Development/kaggle/birdclef


In [2]:
taxonomy = pd.read_csv(DATA_DIR / 'taxonomy.csv')
N_CLASSES = len(taxonomy)
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str)
taxon_ids = taxonomy['primary_label'].tolist()
taxon_to_idx = {tid: i for i, tid in enumerate(taxon_ids)}

perch_labels = pd.read_csv(DATA_DIR / 'models/perch_tf/assets/labels.csv', header=0)
perch_labels.columns = ['scientific_name']
perch_labels['bc_index'] = np.arange(len(perch_labels))

mapping = taxonomy.merge(perch_labels, on='scientific_name', how='left')
comp_to_bc = {
    taxon_to_idx[str(row['primary_label'])]: int(row['bc_index'])
    for _, row in mapping[mapping['bc_index'].notna()].iterrows()
}

def time_to_sec(t):
    h, m, s = map(int, t.split(':'))
    return h * 3600 + m * 60 + s

def parse_soundscape_filename(filename):
    m = re.search(r'BC2026_Train_\d+_(S\d+)_([0-9]{8})_([0-9]{6})\.ogg', filename)
    if not m:
        return pd.Series({'site': 'UNK', 'hour_utc': -1, 'month': -1})
    site, ymd, hms = m.groups()
    return pd.Series({'site': site, 'hour_utc': int(hms[:2]), 'month': int(ymd[4:6])})

raw_df = pd.read_csv(DATA_DIR / 'train_soundscapes_labels.csv')
raw_df['start_sec'] = raw_df['start'].apply(time_to_sec)
labels_df = (
    raw_df
    .drop_duplicates(subset=['filename', 'start_sec'])
    .sort_values(['filename', 'start_sec'])
    .reset_index(drop=True)
)
labels_df = pd.concat([labels_df, labels_df['filename'].apply(parse_soundscape_filename)], axis=1)

def parse_labels(label_str):
    vec = np.zeros(N_CLASSES, dtype=np.float32)
    for tid in str(label_str).split(';'):
        tid = tid.strip()
        if tid in taxon_to_idx:
            vec[taxon_to_idx[tid]] = 1.0
    return vec

Y = np.stack(labels_df['primary_label'].apply(parse_labels).values)
emb_full = np.load(EMB_CACHE).astype(np.float32)
logits_full = np.load(LOG_CACHE).astype(np.float32)
groups = labels_df['filename'].values
folds = list(GroupKFold(n_splits=N_SPLITS).split(emb_full, groups=groups))
print(f'Labels: {Y.shape}, files: {labels_df["filename"].nunique()}')

Labels: (739, 234), files: 66


In [3]:
def build_scores_with_proxy():
    s = np.zeros((len(labels_df), N_CLASSES), dtype=np.float32)
    for comp_idx, bc_idx in comp_to_bc.items():
        s[:, comp_idx] = logits_full[:, bc_idx]
    unmapped = mapping[mapping['bc_index'].isna()].copy()
    for _, row in unmapped.iterrows():
        target = str(row['primary_label'])
        genus = str(row['scientific_name']).split()[0]
        hits = perch_labels[
            perch_labels['scientific_name'].astype(str).str.match(rf'^{re.escape(genus)}\s', na=False)
        ]
        if hits.empty:
            continue
        s[:, taxon_to_idx[target]] = logits_full[:, hits['bc_index'].astype(int).values].max(axis=1)
    return s

scores = build_scores_with_proxy()
raw_probs = sigmoid(scores)
print('Scores:', scores.shape)

Scores: (739, 234)


In [4]:
class MultiTaskMLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return float(np.mean(aps))


def macro_auc(y_true, y_pred):
    aucs = []
    for c in range(y_true.shape[1]):
        pos = y_true[:, c].sum()
        if 0 < pos < len(y_true):
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
    return float(np.mean(aucs)), len(aucs)


def train_one_mlp(X_tr, Y_tr, X_va, Y_va, seed):
    torch.manual_seed(seed)
    model = MultiTaskMLP(X_tr.shape[1], HIDDEN, N_CLASSES, DROPOUT)
    optimizer = torch.optim.Adam(model.parameters(), lr=MLP_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=10, min_lr=1e-5
    )
    pos_counts = Y_tr.sum(axis=0).clip(1)
    pw = np.clip((len(Y_tr) - pos_counts) / pos_counts, 1.0, POS_WEIGHT_CAP).astype(np.float32)
    pos_weight = torch.tensor(pw)
    active_mask = torch.tensor(Y_tr.sum(axis=0) > 0)

    X_tr_t = torch.tensor(X_tr)
    Y_tr_t = torch.tensor(Y_tr)
    X_va_t = torch.tensor(X_va)
    n = len(X_tr_t)
    best_val, best_pred, wait = -1.0, None, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(n)
        for start in range(0, n, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            logits = model(X_tr_t[idx])
            loss = nn.functional.binary_cross_entropy_with_logits(
                logits[:, active_mask], Y_tr_t[idx][:, active_mask],
                pos_weight=pos_weight[active_mask], reduction='mean'
            )
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            vp = sigmoid(model(X_va_t).numpy())
        vs = padded_cmap(Y_va, vp)
        scheduler.step(vs)
        if vs > best_val:
            best_val, best_pred, wait = vs, vp.copy(), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    return best_pred, best_val, epoch + 1

print('Defined.')

Defined.


In [5]:
def fit_prior_tables(meta_df, y_train, ss=8.0, sh=8.0, ssh=4.0):
    gp = y_train.mean(axis=0).astype(np.float32)
    tables = {'global_p': gp, 'site': {}, 'hour': {}, 'site_hour': {}}
    for site, idx in meta_df.groupby('site').groups.items():
        idx = np.array(list(idx), dtype=int)
        tables['site'][str(site)] = ((y_train[idx].sum(0) + ss * gp) / (len(idx) + ss)).astype(np.float32)
    for hour, idx in meta_df.groupby('hour_utc').groups.items():
        idx = np.array(list(idx), dtype=int)
        tables['hour'][int(hour)] = ((y_train[idx].sum(0) + sh * gp) / (len(idx) + sh)).astype(np.float32)
    for (site, hour), idx in meta_df.groupby(['site', 'hour_utc']).groups.items():
        idx = np.array(list(idx), dtype=int)
        tables['site_hour'][(str(site), int(hour))] = ((y_train[idx].sum(0) + ssh * gp) / (len(idx) + ssh)).astype(np.float32)
    return tables

def prior_probs_from_tables(meta_df, tables):
    out = np.repeat(tables['global_p'][None, :], len(meta_df), axis=0).astype(np.float32)
    for i, row in enumerate(meta_df.itertuples(index=False)):
        site, hour = str(row.site), int(row.hour_utc)
        if hour in tables['hour']:
            out[i] = 0.50 * out[i] + 0.50 * tables['hour'][hour]
        if site in tables['site']:
            out[i] = 0.50 * out[i] + 0.50 * tables['site'][site]
        if (site, hour) in tables['site_hour']:
            out[i] = 0.35 * out[i] + 0.65 * tables['site_hour'][(site, hour)]
    return np.clip(out, 1e-5, 1.0 - 1e-5)

def evaluate(name, y_pred, extra=None):
    auc, n_auc = macro_auc(Y, y_pred)
    row = {'method': name, 'macro_auc': auc, 'auc_classes': n_auc, 'padded_cmap': padded_cmap(Y, y_pred)}
    if extra: row.update(extra)
    return row

In [6]:
# Accumulate MLP OOF from all seeds
oof_mlp_seeds = np.zeros((len(MLP_SEEDS),) + raw_probs.shape, dtype=np.float32)
oof_lr = raw_probs.copy()
oof_prior = np.zeros_like(raw_probs)

for fold, (tr, va) in enumerate(folds):
    scaler = StandardScaler()
    pca = PCA(n_components=PCA_DIM, whiten=True, random_state=42)

    emb_tr_sc = scaler.fit_transform(emb_full[tr])
    emb_va_sc = scaler.transform(emb_full[va])

    emb_tr_pca = pca.fit_transform(emb_tr_sc)
    emb_va_pca = pca.transform(emb_va_sc)
    X_tr_lr = np.concatenate([emb_tr_pca, scores[tr]], axis=1)
    X_va_lr = np.concatenate([emb_va_pca, scores[va]], axis=1)

    Y_tr, Y_va = Y[tr], Y[va]

    # --- 3 MLP seeds ---
    seed_preds = []
    for si, seed in enumerate(MLP_SEEDS):
        pred, bv, ep = train_one_mlp(emb_tr_sc, Y_tr, emb_va_sc, Y_va, seed)
        oof_mlp_seeds[si, va] = pred
        seed_preds.append(pred)

    ens_pred = np.mean(seed_preds, axis=0)
    fold_auc, _ = macro_auc(Y_va, ens_pred)

    # --- LR probes ---
    oof_lr[va] = sigmoid(scores[va])
    active = [c for c in range(N_CLASSES) if MIN_POS <= Y_tr[:, c].sum() < len(tr)]
    for c in active:
        clf = LogisticRegression(C=LR_C, class_weight='balanced', max_iter=1000, solver='lbfgs')
        clf.fit(X_tr_lr, Y_tr[:, c])
        oof_lr[va, c] = clf.predict_proba(X_va_lr)[:, 1]

    # --- Priors ---
    train_meta = labels_df.iloc[tr][['site', 'hour_utc']].reset_index(drop=True)
    val_meta = labels_df.iloc[va][['site', 'hour_utc']].reset_index(drop=True)
    oof_prior[va] = prior_probs_from_tables(val_meta, fit_prior_tables(train_meta, Y_tr))

    print(f'Fold {fold+1}: ensemble AUC={fold_auc:.4f} (seeds: {[train_one_mlp.__module__]})')

oof_mlp_ensemble = oof_mlp_seeds.mean(axis=0)
tuned_lr = PROBE_BLEND * oof_lr + (1 - PROBE_BLEND) * raw_probs

print('\nIndividual OOF:')
for name, pred in [('raw_perch', raw_probs), ('oof_mlp_ensemble', oof_mlp_ensemble), ('tuned_lr', tuned_lr)]:
    auc, _ = macro_auc(Y, pred)
    print(f'  {name}: AUC={auc:.4f} cmAP={padded_cmap(Y, pred):.4f}')

Fold 1: ensemble AUC=0.8924 (seeds: ['__main__'])


Fold 2: ensemble AUC=0.8315 (seeds: ['__main__'])


Fold 3: ensemble AUC=0.9091 (seeds: ['__main__'])


Fold 4: ensemble AUC=0.8003 (seeds: ['__main__'])


Fold 5: ensemble AUC=0.8441 (seeds: ['__main__'])

Individual OOF:


  raw_perch: AUC=0.7502 cmAP=0.8666


  oof_mlp_ensemble: AUC=0.8884 cmAP=0.8772


  tuned_lr: AUC=0.8950 cmAP=0.9020


In [7]:
rows = []
for name, pred in [('raw_perch', raw_probs), ('oof_mlp_ensemble', oof_mlp_ensemble), ('tuned_lr', tuned_lr)]:
    rows.append(evaluate(name, pred, {'a_mlp': 0, 'a_lr': 0, 'a_perch': 0, 'prior_blend': 0}))

for a_mlp in A_MLP_GRID:
    for a_lr in A_LR_GRID:
        a_perch = round(1.0 - a_mlp - a_lr, 6)
        if a_perch < 0 or a_perch > 1:
            continue
        blended = a_mlp * oof_mlp_ensemble + a_lr * tuned_lr + a_perch * raw_probs
        rows.append(evaluate('3way', blended, {'a_mlp': a_mlp, 'a_lr': a_lr, 'a_perch': a_perch, 'prior_blend': 0.0}))
        for pb in PRIOR_BLEND_GRID:
            if pb == 0:
                continue
            rows.append(evaluate('3way_prior', (1 - pb) * blended + pb * oof_prior,
                                 {'a_mlp': a_mlp, 'a_lr': a_lr, 'a_perch': a_perch, 'prior_blend': pb}))

results = pd.DataFrame(rows).sort_values(['padded_cmap', 'macro_auc'], ascending=False).reset_index(drop=True)
results.to_csv(RESULTS_CSV, index=False)

print('Top 15:')
print(results.head(15).to_string(index=False, float_format=lambda x: f'{x:.4f}'))

best = results.iloc[0].to_dict()
best['exp09_baseline'] = {'macro_auc': 0.9176, 'padded_cmap': 0.9144}
with open(RESULTS_JSON, 'w') as f:
    json.dump(best, f, indent=2)
print('\nSaved:', RESULTS_JSON)

Top 15:
    method  macro_auc  auc_classes  padded_cmap  a_mlp   a_lr  a_perch  prior_blend
3way_prior     0.9119           75       0.9099 0.1000 0.3000   0.6000       0.1000
3way_prior     0.9166           75       0.9097 0.1500 0.3000   0.5500       0.1000
3way_prior     0.9177           75       0.9096 0.1500 0.4000   0.4500       0.1000
3way_prior     0.9131           75       0.9096 0.1000 0.4000   0.5000       0.1000
3way_prior     0.9139           75       0.9096 0.1000 0.5000   0.4000       0.1000
3way_prior     0.9143           75       0.9096 0.1000 0.6000   0.3000       0.1000
3way_prior     0.9065           75       0.9096 0.0500 0.3000   0.6500       0.1000
3way_prior     0.9182           75       0.9094 0.1500 0.5000   0.3500       0.1000
3way_prior     0.9168           75       0.9094 0.1500 0.3000   0.5500       0.0750
3way_prior     0.9185           75       0.9094 0.1500 0.6000   0.2500       0.1000
3way_prior     0.9141           75       0.9094 0.1000 0.5000   0.40

## Result Notes
Exp09 baseline: OOF AUC 0.9176, padded cmAP 0.9144 (best: 10% MLP + 60% tuned_LR + 30% raw Perch + 5% prior).